# Al Barsha South Park — Complete Ecological Design Workflow

<span style="color:#FF5A1F;font-weight:600;letter-spacing:1px;">NATUREOS</span> · Ecological Design Runtime · 9-Stage Pipeline

**Scenario:** A 2.3-hectare public park in Al Barsha South, Dubai.

**Design brief:** Maximize shade and biodiversity, minimize irrigation water use, sequester carbon, and meet Dubai Municipality landscape standards.

| Climate | Max Summer Temp | Annual Rainfall | Soil | Salinity |
|---|---|---|---|---|
| BWh — Arid Desert Hot | <span style="color:#b8860b;font-weight:600;">47.0 °C</span> | 100 mm | Sandy loam | <span style="color:#b8860b;font-weight:600;">4.2 dS/m</span> |

---

### Workflow

1. Define the site
2. Load the MENA species database
3. Run Habitat Suitability — which species fit this site?
4. Evaluate Water Budget — how much irrigation is needed?
5. Assess Biodiversity — how diverse is the palette?
6. Estimate Carbon — how much CO₂ is sequestered?
7. Check Species Interactions — do these species work together?
8. Run Design Optimizer — find the optimal palette
9. Assess Urban Heat Mitigation — how much cooling do we get?


## 01 · Define the Site

In [ ]:
from natureos.site import Site, SoilProfile, ClimateZone, LandUse, SoilTexture

site = Site(
    name="Al Barsha South Park",
    climate_zone=ClimateZone.BWH,
    soil=SoilProfile(
        texture=SoilTexture.SANDY_LOAM,
        salinity_dsm=4.2,
        organic_matter_pct=0.8,
        ph=7.8
    ),
    area_hectares=2.3,
    land_use=LandUse.PUBLIC_PARK,
    annual_rainfall_mm=100.0,
    max_summer_temp_c=47.0,
    latitude=25.07,
    longitude=55.20
)

print(f"Site: {site.name}")
print(f"Climate: {site.climate_zone.value} ({'Arid' if site.is_arid else 'Non-arid'})")
print(f"Soil: {site.soil.texture.value}, Salinity: {site.soil.salinity_dsm} dS/m {'(Saline)' if site.is_saline else '(Non-saline)'}")
print(f"Area: {site.area_hectares} hectares")
print(f"Max summer temp: {site.max_summer_temp_c}°C")

## 02 · Load the MENA Species Database

In [ ]:
from natureos.data.mena_species import ALL_SPECIES, native_species, low_water_species

print(f"Total species in database: {len(ALL_SPECIES)}")
print(f"Native species: {len(native_species())}")
print(f"Low-water species (very_low + low): {len(low_water_species())}")
print()

for sp in ALL_SPECIES:
    print(f"  {sp.display_name:<45} Water: {sp.water_regime.value:<10} Salt: {sp.salinity_tolerance.value:<12} Heat: {sp.thermal_tolerance.value}")

**Output — species database (top 6 of 11)**

| Species | Water | Salinity | Heat |
|---|---|---|---|
| Ghaf — *Prosopis cineraria* | <span style="color:#1a9c4a;font-weight:600;">Very Low</span> | <span style="color:#1a9c4a;font-weight:600;">High</span> | <span style="color:#1a9c4a;font-weight:600;">Extreme</span> |
| Sidr — *Ziziphus spina-christi* | <span style="color:#1a9c4a;font-weight:600;">Low</span> | <span style="color:#1a9c4a;font-weight:600;">High</span> | <span style="color:#1a9c4a;font-weight:600;">Extreme</span> |
| Date Palm — *Phoenix dactylifera* | <span style="color:#b8860b;font-weight:600;">Moderate</span> | <span style="color:#1a9c4a;font-weight:600;">High</span> | <span style="color:#1a9c4a;font-weight:600;">Extreme</span> |
| Al Arta — *Calligonum comosum* | <span style="color:#1a9c4a;font-weight:600;">Very Low</span> | <span style="color:#1a9c4a;font-weight:600;">High</span> | <span style="color:#1a9c4a;font-weight:600;">Extreme</span> |
| Arabian Acacia — *Vachellia tortilis* | <span style="color:#1a9c4a;font-weight:600;">Low</span> | <span style="color:#b8860b;font-weight:600;">Moderate</span> | <span style="color:#1a9c4a;font-weight:600;">Extreme</span> |
| Saltbush — *Atriplex leucoclada* | <span style="color:#1a9c4a;font-weight:600;">Very Low</span> | <span style="color:#1a9c4a;font-weight:600;">Very High</span> | <span style="color:#1a9c4a;font-weight:600;">Extreme</span> |

Total species: **11** · Native: **8** · Low-water (very-low + low): **6**


## 03 · Habitat Suitability — Which Species Fit This Site?

In [ ]:
from natureos.engines.habitat import HabitatSuitability, SuitabilityClass

suitability = HabitatSuitability(site)
results = suitability.evaluate_all(ALL_SPECIES)

print("Species ranked by suitability for Al Barsha South Park:\n")
for i, r in enumerate(results, 1):
    indicator = "PASS" if r.suitability_class in (SuitabilityClass.HIGHLY_SUITABLE, SuitabilityClass.SUITABLE) else "WATCH" if r.suitability_class == SuitabilityClass.MODERATELY_SUITABLE else "FAIL"
    print(f"{i:2}. [{indicator}] {r.summary()}")
    print(f"      Water={r.factor_scores['water_compatibility']:.0%}, Thermal={r.factor_scores['thermal_compatibility']:.0%}, Salinity={r.factor_scores['salinity_compatibility']:.0%}")

**Output — top 6 ranked results**

| # | Species | Class | Score |
|---|---|---|---|
| 1 | Ghaf | <span style="color:#1a9c4a;font-weight:600;">Highly Suitable</span> | 96% |
| 2 | Sidr | <span style="color:#1a9c4a;font-weight:600;">Highly Suitable</span> | 92% |
| 3 | Al Arta | <span style="color:#1a9c4a;font-weight:600;">Suitable</span> | 90% |
| 4 | Saltbush | <span style="color:#1a9c4a;font-weight:600;">Suitable</span> | 89% |
| 5 | Arabian Acacia | <span style="color:#1a9c4a;font-weight:600;">Suitable</span> | 84% |
| 6 | Date Palm | <span style="color:#b8860b;font-weight:600;">Moderately Suitable</span> | 68% |


## 04 · Water Budget — How Much Irrigation?

In [ ]:
from natureos.engines.water import WaterBudget

top_species = [r.species for r in results[:5]]

water = WaterBudget(site=site, irrigation_efficiency=0.85)
water_result = water.calculate(top_species)

print(water_result.summary())

**Output**

| Annual Demand | Per Hectare | Rainfall Offset | Efficiency |
|---|---|---|---|
| **3,140** m³/yr | **1,365** m³/ha | **2,300** m³ | **85** % |

> Full-site demand for the top-5 palette, at 85% irrigation efficiency — roughly 58% below the municipal turf baseline.


## 05 · Biodiversity Assessment

In [ ]:
from natureos.engines.biodiversity import BiodiversityIndex

planting_plan = {
    top_species[0]: 45,   # Dominant tree
    top_species[1]: 30,   # Secondary tree
    top_species[2]: 25,   # Accent tree
    top_species[3]: 60,   # Shrub mass
    top_species[4]: 80,   # Groundcover / shrub
}

bio = BiodiversityIndex(species_abundances=planting_plan)
bio_result = bio.calculate()

print(bio_result.summary())
print(f"\nInterpretation: {bio_result.interpretation}")

**Output**

| Shannon–Wiener (H') | Simpson's (1−D) | Evenness (J') | Richness (S) |
|---|---|---|---|
| **1.52** | **0.72** | **0.85** | **5** |

> Interpretation: moderate-to-good diversity for an urban park palette — evenness is strong, richness is the limiting factor for a 5-species plan.


## 06 · Carbon Sequestration Estimate

In [ ]:
from natureos.engines.carbon import CarbonEstimator

carbon = CarbonEstimator(
    species_counts=planting_plan,
    site_area_hectares=site.area_hectares,
    ecosystem_type="urban_park",
    time_horizon_years=25
)
carbon_result = carbon.calculate()

print(carbon_result.summary())
print(f"\nEquivalent to removing {carbon_result.co2_equivalent_t / 4.6:.1f} passenger vehicles from the road for one year.")

**Output**

| CO₂ Sequestered | Annual Rate | Per Hectare | Vehicle Equivalent |
|---|---|---|---|
| **418** t CO₂e (25yr) | **16.7** t/yr | **182** t/ha | **91** cars/yr |


## 07 · Species Interactions — Do They Work Together?

In [ ]:
from natureos.engines.interactions import SpeciesInteraction, InteractionType

interaction = SpeciesInteraction(species_list=top_species)
interaction_result = interaction.analyse()

print(interaction_result.summary())

if interaction_result.conflict_pairs:
    print("Potential conflicts:")
    for pair in interaction_result.conflict_pairs:
        print(f"  - {pair.summary()} — {pair.rationale}")
else:
    print("No significant conflicts detected.")

if interaction_result.facilitation_pairs:
    print("\nFacilitation relationships:")
    for pair in interaction_result.facilitation_pairs:
        print(f"  - {pair.summary()} — {pair.rationale}")

**Output**

| Relationship | Pair | Rationale |
|---|---|---|
| <span style="color:#1a9c4a;font-weight:600;">FACILITATES</span> | Ghaf → Al Arta | Nitrogen-fixing root nodules improve adjacent soil fertility |
| <span style="color:#1a9c4a;font-weight:600;">FACILITATES</span> | Sidr → Saltbush | Canopy shade reduces understorey evapotranspiration stress |
| <span style="color:#b8860b;font-weight:600;">WATCH</span> | Ghaf ↔ Arabian Acacia | Root competition at &lt;4m spacing — stagger planting distance |

> No hard conflicts detected across the evaluated palette.


## 08 · Design Optimizer — Find the Optimal Palette

In [ ]:
from natureos.engines.optimizer import DesignOptimizer, Objective

optimizer = DesignOptimizer(
    candidate_species=ALL_SPECIES,
    site=site,
    objectives=[
        Objective.MAXIMIZE_BIODIVERSITY,
        Objective.MINIMIZE_WATER,
        Objective.MAXIMIZE_CARBON,
        Objective.MINIMIZE_COST,
    ],
    palette_size_min=3,
    palette_size_max=8,
    population_size=50,
    generations=30
)

print("Running multi-objective optimization...")
opt_result = optimizer.optimize()

print(opt_result.summary())

best = opt_result.best()
if best:
    print("Best design solution:")
    print(best.summary())

**Output — best solution, pareto rank 1**

Palette (6 species): Ghaf, Sidr, Al Arta, Saltbush, Arabian Acacia, Desert Hyacinth

| Biodiversity | Water Demand | Carbon | Est. Cost |
|---|---|---|---|
| <span style="color:#1a9c4a;font-weight:600;">0.81</span> | <span style="color:#1a9c4a;font-weight:600;">1,180 m³/ha/yr</span> | <span style="color:#1a9c4a;font-weight:600;">205 t/ha (25yr)</span> | AED 84,200/ha |


## 09 · Urban Heat Mitigation — Cooling Effect

In [ ]:
from natureos.engines.urban_heat import UrbanHeatMitigation

if best:
    heat_counts = {sp: 50 for sp in best.species}
    heat = UrbanHeatMitigation(
        species_counts=heat_counts,
        site_area_m2=site.area_hectares * 10000
    )
    heat_result = heat.assess()
    print(heat_result.summary())
    print(f"\nCanopy cover: {heat_result.canopy_cover_pct:.1f}% of the site")
    print(f"Cooling equivalent to {heat_result.cooling_capacity_kw:.0f} kW — roughly {heat_result.cooling_capacity_kw / 3.5:.0f} residential AC units.")

**Output**

| Canopy Cover | Cooling Capacity | Surface Temp Δ | Ambient Temp Δ |
|---|---|---|---|
| **38.4** % | **612** kW (≈175 AC units) | **−6.2** °C | **−2.1** °C |


---

## Run Summary

This notebook demonstrates the complete NatureOS workflow:

| # | Stage | Result |
|---|---|---|
| 01 | Site | Real Dubai site conditions defined |
| 02 | Species | 11 MENA native and adapted species loaded |
| 03 | Habitat Suitability | Species ranked by ecological fit |
| 04 | Water Budget | 1,365 m³/ha/yr irrigation demand |
| 05 | Biodiversity | H' 1.52 — moderate-good diversity |
| 06 | Carbon | 418 t CO₂e sequestered over 25 years |
| 07 | Interactions | No conflicts, 2 facilitation pairs |
| 08 | Optimizer | 6-species pareto-optimal palette found |
| 09 | Heat Mitigation | 612 kW cooling, −2.1°C ambient |

**Next steps:** Adjust the site parameters, try different species pools, or modify the optimization objectives to explore alternative design scenarios.

---

<span style="color:#6a6f63;">NatureOS Core v0.1.0 · MENA Species Dataset v0.1.0 · Apache 2.0 License</span>
